# SIDRM Quick Start Guide

This notebook demonstrates the complete workflow for training and evaluating the **Smart Interpretable Dietary Recommender Model (SIDRM)** for vulnerable populations.

**Paper**: "Smart Interpretable Dietary Recommender Model for Vulnerable Populations"  
**Authors**: Zvinodashe Revesai and Okuthe P. Kogeda (2025)

## 1. Setup and Imports

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.models.sidrm import SIDRM
from src.models.sidrm_mobile import SIDRMMobileOptimized, SIDRMEdgeOnly
from src.data.nhanes_dataset import NHANESPreprocessor, create_nhanes_dataloaders
from src.training.sidrm_trainer import SIDRMTrainer
from src.training.sidrm_losses import SIDRMMultiObjectiveLoss
from src.evaluation.sidrm_metrics import SIDRMEvaluator
from src.evaluation.sidrm_interpretability import SIDRMInterpretabilityEvaluator

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Device configuration
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

## 2. Data Preparation

Generate NHANES-style data for vulnerable populations:  
- Pregnant women (P)  
- Elderly individuals (E)  
- Children (C)  
- Chronic disease patients (CD)

In [ ]:
# Initialize preprocessor
preprocessor = NHANESPreprocessor(random_state=42)

# Generate synthetic NHANES data (replace with real data loading in production)
data_splits = preprocessor.generate_synthetic_nhanes_data(
    n_samples=2000,  # Using smaller dataset for demo
    train_ratio=0.7,
    val_ratio=0.15
)

# Create dataloaders
dataloaders = create_nhanes_dataloaders(
    data_splits,
    preprocessor,
    batch_size=32,
    num_workers=0  # Set to 0 for notebooks
)

print(f"\nDataset sizes:")
print(f"  Training: {len(dataloaders['train'].dataset)}")
print(f"  Validation: {len(dataloaders['val'].dataset)}")
print(f"  Test: {len(dataloaders['test'].dataset)}")

## 3. Model Creation

Create SIDRM model with architecture from the paper:  
- 5 hierarchical transformer layers  
- 8-head multi-head attention  
- Population-specific encoders  
- Cross-population integration

In [ ]:
# Get data dimensions
batch = next(iter(dataloaders['train']))
input_dim = batch['features'].shape[1]
num_nutrients = batch['labels'].shape[1]

print(f"Input features: {input_dim}")
print(f"Number of nutrients: {num_nutrients}")

# Create SIDRM model
model = SIDRM(
    input_dim=input_dim,
    num_populations=4,
    population_names=['Pregnant', 'Elderly', 'Children', 'Chronic_Disease'],
    hidden_dim=128,
    num_layers=5,
    num_attention_heads=8,
    num_nutrients=num_nutrients,
    dropout_rate=0.3,
    use_population_specific=True
)

print("\n" + model.get_model_summary())

## 4. Multi-Objective Loss Function

Implement Equation (13):  
$$L_{total} = 0.4 \cdot L_{accuracy} + 0.3 \cdot L_{interpret} + 0.2 \cdot L_{clinical} + 0.1 \cdot L_{safety}$$

In [ ]:
# Create multi-objective loss
criterion = SIDRMMultiObjectiveLoss(
    num_nutrients=num_nutrients,
    alpha=0.4,  # Accuracy
    beta=0.3,   # Interpretability
    gamma=0.2,  # Clinical
    delta=0.1   # Safety
)

print("Loss function weights:")
print(criterion.get_loss_weights())

## 5. Training

Train with AdamW optimizer (learning rate 1e-4, weight decay 1e-5)

In [ ]:
# Create trainer
trainer = SIDRMTrainer(
    model=model,
    criterion=criterion,
    train_loader=dataloaders['train'],
    val_loader=dataloaders['val'],
    learning_rate=1e-4,
    weight_decay=1e-5,
    device=device,
    patience=15,
    decay_factor=0.7,
    save_dir='../checkpoints/sidrm_demo'
)

# Train for 10 epochs (increase for better results)
history = trainer.train(
    num_epochs=10,
    early_stopping=True,
    early_stopping_patience=5,
    save_best=True,
    verbose=True
)

## 6. Training Visualization

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss plot
axes[0].plot(history['train_loss'], label='Training Loss')
axes[0].plot(history['val_loss'], label='Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy plot
axes[1].plot(history['train_accuracy'], label='Training Accuracy')
axes[1].plot(history['val_accuracy'], label='Validation Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Model Evaluation

Comprehensive evaluation including per-population and per-nutrient metrics

In [ ]:
# Create evaluator
evaluator = SIDRMEvaluator(
    model=model,
    device=device,
    population_names=['Pregnant', 'Elderly', 'Children', 'Chronic_Disease'],
    nutrient_names=preprocessor.nutrient_names
)

# Evaluate on test set
results = evaluator.evaluate(
    dataloaders['test'],
    return_predictions=True,
    compute_per_nutrient=True,
    compute_per_population=True
)

# Print results
evaluator.print_evaluation_results(results, detailed=True)

## 8. Interpretability Analysis

Compute SHAP stability, attention consistency, and feature ranking correlation

In [ ]:
# Get background data for SHAP
train_batch = next(iter(dataloaders['train']))
background_data = train_batch['features'][:50]

# Create interpretability evaluator
interp_evaluator = SIDRMInterpretabilityEvaluator(
    model=model,
    background_data=background_data,
    feature_names=preprocessor.feature_names,
    num_bootstrap_samples=5,  # Reduced for demo
    device=device
)

# Evaluate interpretability (without SHAP for speed)
test_batch = next(iter(dataloaders['test']))
test_features = test_batch['features'][:20]
test_populations = test_batch['population'][:20]

interp_results = interp_evaluator.evaluate_interpretability(
    test_features,
    test_populations,
    compute_shap=False,  # Set to True for full evaluation (slow)
    shap_nsamples=50
)

print("\nInterpretability Metrics:")
print(f"Attention Consistency: {interp_results['attention_consistency']:.4f}")
print(f"Overall Interpretability: {interp_results['overall_interpretability']:.4f}")

## 9. Mobile-Optimized Models

Test mobile and edge configurations for deployment

In [ ]:
# Mobile-Optimized SIDRM (81% size reduction)
mobile_model = SIDRMMobileOptimized(
    input_dim=input_dim,
    num_populations=4,
    hidden_dim=64,
    num_layers=3,
    num_attention_heads=4,
    num_nutrients=num_nutrients
)

mobile_size = mobile_model.get_model_size()
print("Mobile-Optimized Model:")
print(f"  Size: {mobile_size['size_mb']:.2f} MB")
print(f"  Parameters: {mobile_size['parameters']:,}")
print(f"  Target: 32.8 MB (Table 4)\n")

# Edge-Only SIDRM
edge_model = SIDRMEdgeOnly(
    input_dim=input_dim,
    hidden_dim=48,
    num_layers=2,
    num_nutrients=num_nutrients
)

edge_size = edge_model.get_model_size()
print("Edge-Only Model:")
print(f"  Size: {edge_size['size_mb']:.2f} MB")
print(f"  Parameters: {edge_size['parameters']:,}")
print(f"  Target: 28.1 MB (Table 4)")

## 10. Per-Population Performance Visualization

In [ ]:
# Visualize per-population performance
if 'per_population' in results:
    pop_names = list(results['per_population'].keys())
    accuracies = [results['per_population'][pop]['accuracy'] * 100 for pop in pop_names]
    f1_scores = [results['per_population'][pop]['f1_score'] for pop in pop_names]
    
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Accuracy by population
    axes[0].bar(pop_names, accuracies, color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A'])
    axes[0].set_ylabel('Accuracy (%)')
    axes[0].set_title('Accuracy by Vulnerable Population')
    axes[0].set_ylim([0, 100])
    axes[0].grid(True, alpha=0.3, axis='y')
    
    # F1-Score by population
    axes[1].bar(pop_names, f1_scores, color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A'])
    axes[1].set_ylabel('F1-Score')
    axes[1].set_title('F1-Score by Vulnerable Population')
    axes[1].set_ylim([0, 1])
    axes[1].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()

## 11. Model Comparison (Table 3)

Compare SIDRM with baseline models

In [ ]:
import pandas as pd

# Results from paper (Table 3)
comparison_data = [
    {'Model': 'SIDRM (Ours)', 'Accuracy (%)': 91.0, 'F1-Score': 0.89, 'Interpretability': 0.89},
    {'Model': 'Transformer Baseline', 'Accuracy (%)': 87.0, 'F1-Score': 0.85, 'Interpretability': 0.41},
    {'Model': 'CNN-Based', 'Accuracy (%)': 85.3, 'F1-Score': 0.83, 'Interpretability': 0.31},
    {'Model': 'Rule-Based', 'Accuracy (%)': 76.0, 'F1-Score': 0.74, 'Interpretability': 0.93},
]

comparison_df = pd.DataFrame(comparison_data)
print("\nModel Comparison (Table 3):")
print(comparison_df.to_string(index=False))

# Visualization
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(comparison_df))
width = 0.25

ax.bar(x - width, comparison_df['Accuracy (%)'], width, label='Accuracy (%)', color='#4ECDC4')
ax.bar(x, comparison_df['F1-Score'] * 100, width, label='F1-Score (×100)', color='#45B7D1')
ax.bar(x + width, comparison_df['Interpretability'] * 100, width, label='Interpretability (×100)', color='#FFA07A')

ax.set_xlabel('Model')
ax.set_ylabel('Score')
ax.set_title('SIDRM vs. State-of-the-Art Models')
ax.set_xticks(x)
ax.set_xticklabels(comparison_df['Model'], rotation=15, ha='right')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated:

1. ✅ **Data preparation** for NHANES dataset (4 vulnerable populations)
2. ✅ **Model creation** with population-specific processing
3. ✅ **Multi-objective training** (Equation 13)
4. ✅ **Comprehensive evaluation** (accuracy, per-population, per-nutrient)
5. ✅ **Interpretability analysis** (SHAP, attention consistency)
6. ✅ **Mobile deployment** configurations (81% size reduction)

### Key Results

- **Overall Accuracy**: 91.0% (vs 87.0% baseline)
- **SHAP Stability**: 0.91
- **Attention Consistency**: 0.93
- **Parameter Reduction**: 64% (127.3M → 56.2M)
- **Mobile Size**: 32.8 MB (81% reduction from 174.9 MB)

### Next Steps

1. Train on real NHANES data (2015-2018 cycles)
2. Extend training to 200 epochs for optimal performance
3. Validate with registered dietitians
4. Deploy to mobile platforms (iOS/Android)
5. Conduct clinical trials with vulnerable populations